In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

for name in ["WANDB_API_KEY", "HF_TOKEN"]:
    if value := secret(name):
        os.environ[name] = value
if "WANDB_API_KEY" not in os.environ:  # run pushed via the Kaggle CLI has no Secrets: log offline, `wandb sync` later
    os.environ["WANDB_MODE"] = "offline"

REPO_DIR = "/kaggle/working/ai500-labs"
BRANCH = "main"
url = "https://github.com/PavloKhomenko/ai500-labs.git"

In [ ]:
# the token is passed only on the command line, origin keeps the public url (/kaggle/working gets saved)
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q {url} {BRANCH}
else:
    !git clone -q -b {BRANCH} {url} {REPO_DIR}
    !git -C {REPO_DIR} remote set-url origin https://github.com/PavloKhomenko/ai500-labs.git
!git -C {REPO_DIR} log --oneline -1

In [ ]:
# versions from lab-1/uv.lock; torch stays Kaggle's CUDA build
!pip install -q "transformers==5.18.0" "datasets==5.1.0" "accelerate==1.15.0" "peft==0.21.2" \
    "huggingface-hub==1.33.0" "wandb==0.30.0" "tabulate==0.10.0" "pyyaml==6.0.3"
# Kaggle ships torchao 0.10; peft 0.21 needs torchao > 0.16 and errors while dispatching LoRA layers, and we don't use it
!pip uninstall -q -y torchao

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/lab-1/src")

import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer

from lab1.common import load_config, out_dir
from lab1.data import prepare_part1_data
from lab1.part2 import wandb_log as wl
from lab1.part2.pretrain import load_model
from lab1.part1.evaluate import LongContextEvaluator, ResultsReport
from lab1.part1.finetune import LongLoRATrainer
%matplotlib inline

cfg = load_config()
p1 = cfg["part1"]
device = torch.device("cuda")
env = {"torch": torch.__version__, "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name()}
print(env)

def start_run(name, job_type, **extra):
    return wl.init_run(cfg, name, job_type, config={**p1, **env, **extra}, group="part1",
                       project=cfg["wandb"]["part1_project"])

In [ ]:
import glob
import shutil
from pathlib import Path

# input: dataset khomenkopavlo/lab1-part1-outputs (flat files from the lab1-part1 output)
found = glob.glob("/kaggle/input/**/data.pt", recursive=True)
assert found, "add the lab1-part1-outputs dataset as an Input"
prev = Path(found[0]).parent
for f in prev.iterdir():
    sub = "data" if f.name in ("data.pt", "eval_doc_keys.json") else "kv_cache" if f.stem == "kv_cache" else "eval"
    shutil.copy(f, out_dir(cfg, "part1", sub) / f.name)
print("restored from", prev)

tok = AutoTokenizer.from_pretrained(p1["model"])
data = prepare_part1_data(cfg, tok, out_dir(cfg, "part1", "data"))
eval_dir, fig_dir = out_dir(cfg, "part1", "eval"), out_dir(cfg, "part1", "figures")
print({k: tuple(v.shape) for k, v in data.items() if torch.is_tensor(v)})
print(sorted(f.name for f in eval_dir.glob("*.json")))

In [ ]:
ft_dir = out_dir(cfg, "part1", "finetune")

start_run(f"finetune-{p1['finetune']['method']}", "train", **p1["finetune"])
trainer = LongLoRATrainer(load_model(p1["model"], device), p1, seed=cfg["seed"])
trainer.train(data["train"], ft_dir / "train_log.jsonl", max_steps=None)  # max_steps=5 for timing
trainer.save(ft_dir / "trainable.pt")
wl.finish()

In [ ]:
del trainer
torch.cuda.empty_cache()
method = p1["finetune"]["method"]
model = LongLoRATrainer.load_finetuned(load_model(p1["model"], device), ft_dir / "trainable.pt", p1)
evaluator = LongContextEvaluator(model, tok, p1, seed=cfg["seed"])

start_run(f"eval-{method}-ft", "eval", method=method, finetuned=True)
evaluator.evaluate(method, data, eval_dir, tag=f"{method}-ft")
wl.finish()

In [ ]:
report = ResultsReport(eval_dir, fig_dir, p1["l_train"], p1["bin_size"])
start_run("eval-summary", "analysis")
report.loss_plot("uk")
report.loss_plot("en")
bpb, passkey = report.bpb_table(), report.passkey_table()
wl.finish()
display(bpb, passkey)